# 23CSE301 Machine Learning - Capstone Project
## Clustering Track - Review 2 (Complete - Both Algorithms)

**Team:** TASKMASTERS (Team 16)  
**Dataset:** Australian Open / PGA 2022  
**Task:** Unsupervised Clustering Analysis

---

### Review 2 Requirements - Clustering Track

This notebook implements **BOTH clustering algorithms** as specified in rubric section 3.3:

1. **K-Means Clustering** - Plot Elbow curve (inertia vs. k) to choose k
2. **Agglomerative Hierarchical Clustering** - Plot Dendrogram; compare linkage strategies

### Evaluation Metrics (All Mandatory):
- **Silhouette Score** - Higher is better (range: -1 to 1)
- **Davies-Bouldin Index** - Lower is better
- **Calinski-Harabasz Index** - Higher is better

### Required Visualizations:
- Elbow curve for K-Means (MANDATORY)
- Dendrogram for Hierarchical clustering (MANDATORY)
- PCA 2D scatter plot for BOTH algorithms (MANDATORY)
- t-SNE visualization for at least ONE algorithm (ENCOURAGED)

### Important Note:
**Per rubric:** "Unlike regression and classification, clustering has no labels during training. Use ground-truth labels ONLY to interpret or validate results AFTER the fact, not during model fitting." 

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn - Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Clustering Algorithms (per rubric section 3.3)
from sklearn.cluster import KMeans, AgglomerativeClustering

# Dimensionality Reduction for Visualization
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Clustering Evaluation Metrics (all mandatory)
from sklearn.metrics import (
    silhouette_score,          # Mandatory
    davies_bouldin_score,      # Mandatory
    calinski_harabasz_score    # Mandatory
)

# Hierarchical clustering visualization
from scipy.cluster.hierarchy import dendrogram, linkage

# Random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('All libraries imported successfully')
print(f'Random state: {RANDOM_STATE}')

## 1. Data Loading and Exploration

In [ ]:
# Load dataset - you can choose either dataset
# Option 1: Australian Open (classification dataset without labels)
# Option 2: PGA 2022 (regression dataset without target)

# Using Australian Open for clustering
df = pd.read_csv('AustralianOpen.csv')

print(f'Dataset shape: {df.shape}')
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Dataset information
print('Dataset Info:')
df.info()

print('\nMissing values:')
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print('No missing values')

print('\nBasic statistics:')
df.describe()

## 2. Data Preprocessing for Clustering

**Key Steps:**
- Remove target/label column (clustering is unsupervised)
- Select numerical features only
- Handle missing values
- Standardize features (important for distance-based clustering)

In [ ]:
# Remove label column if present and select numeric features
# Identify potential label columns
label_cols = ['Won', 'avgScore', 'target', 'label', 'class']
cols_to_drop = [col for col in label_cols if col in df.columns]

# Keep only numerical features for clustering
X = df.drop(cols_to_drop, axis=1).select_dtypes(include=[np.number])

print(f'Features for clustering: {X.shape[1]} numerical features')
print(f'Samples: {X.shape[0]}')
print(f'\nFeatures: {list(X.columns[:10])}...')  # Show first 10

# Store ground truth if available (for validation only, not training)
if 'Won' in df.columns:
    y_true = df['Won']
    print(f'\nGround truth labels available: {y_true.nunique()} classes')
    print('(Note: Will be used ONLY for validation, not during clustering)')
else:
    y_true = None
    print('\nNo ground truth labels (pure unsupervised clustering)')

In [ ]:
# Handle missing values and standardize
# Missing values
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Standardization (CRITICAL for K-Means and distance-based clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print('Preprocessing complete')
print(f'Scaled data shape: {X_scaled.shape}')
print(f'Mean: {X_scaled.mean():.2e} (should be ~0)')
print(f'Std: {X_scaled.std():.2f} (should be ~1)')

## 3. K-Means Clustering

**Per rubric:** "Plot Elbow curve (inertia vs. k) to choose k" 

### Algorithm Steps:
1. Try different values of k (2 to 10)
2. Plot elbow curve to find optimal k
3. Calculate silhouette scores for each k
4. Select optimal k
5. Fit final K-Means model
6. Calculate all mandatory metrics

In [ ]:
# Elbow Method to find optimal k
inertias = []
silhouette_scores = []
K_range = range(2, 11)

print('Running K-Means for different k values...')
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    
    # Calculate silhouette score
    labels = kmeans.labels_
    sil_score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil_score)
    
    print(f'  k={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={sil_score:.4f}')

print('\nElbow analysis complete')

In [ ]:
# Plot Elbow Curve (MANDATORY visualization per rubric)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12, fontweight='bold')
axes[0].set_title('Elbow Method For Optimal K', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].set_xticks(K_range)

# Silhouette plot
axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_title('Silhouette Score vs Number of Clusters', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].set_xticks(K_range)

plt.tight_layout()
plt.show()

# Suggest optimal k
optimal_k_silhouette = K_range[np.argmax(silhouette_scores)]
print(f'\nRecommended k based on highest silhouette score: {optimal_k_silhouette}')
print(f'Silhouette score: {max(silhouette_scores):.4f}')

In [ ]:
# Fit final K-Means model with optimal k
optimal_k = 3  # You can change this based on elbow curve

print(f'Fitting K-Means with k={optimal_k}...')
kmeans_final = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_scaled)

# Calculate all mandatory metrics
sil_kmeans = silhouette_score(X_scaled, kmeans_labels)
db_kmeans = davies_bouldin_score(X_scaled, kmeans_labels)
ch_kmeans = calinski_harabasz_score(X_scaled, kmeans_labels)

print(f'\nK-Means Results (k={optimal_k}):')
print(f'  Silhouette Score: {sil_kmeans:.4f} (higher is better)')
print(f'  Davies-Bouldin Index: {db_kmeans:.4f} (lower is better)')
print(f'  Calinski-Harabasz Index: {ch_kmeans:.2f} (higher is better)')

print(f'\nCluster sizes:')
unique, counts = np.unique(kmeans_labels, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f'  Cluster {cluster}: {count} samples ({count/len(kmeans_labels)*100:.1f}%)')

## 4. Agglomerative Hierarchical Clustering

**Per rubric:** "Plot Dendrogram; compare linkage strategies" 

### Linkage Methods to Compare:
- **Ward** - Minimizes variance of merged clusters
- **Complete** - Maximum distance between clusters
- **Average** - Average distance between clusters
- **Single** - Minimum distance between clusters

In [ ]:
# Compare different linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
linkage_results = {}

print(f'Testing hierarchical clustering with {optimal_k} clusters...')
print('\nLinkage method comparison:')

for method in linkage_methods:
    hier = AgglomerativeClustering(n_clusters=optimal_k, linkage=method)
    hier_labels = hier.fit_predict(X_scaled)
    
    # Calculate metrics
    sil = silhouette_score(X_scaled, hier_labels)
    db = davies_bouldin_score(X_scaled, hier_labels)
    ch = calinski_harabasz_score(X_scaled, hier_labels)
    
    linkage_results[method] = {
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'Calinski-Harabasz': ch,
        'labels': hier_labels
    }
    
    print(f'  {method:10s}: Silhouette={sil:.4f}, DB={db:.4f}, CH={ch:.2f}')

# Select best linkage method
best_linkage = max(linkage_results, key=lambda x: linkage_results[x]['Silhouette'])
print(f'\nBest linkage method: {best_linkage} (highest silhouette score)')

hier_labels = linkage_results[best_linkage]['labels']
sil_hier = linkage_results[best_linkage]['Silhouette']
db_hier = linkage_results[best_linkage]['Davies-Bouldin']
ch_hier = linkage_results[best_linkage]['Calinski-Harabasz']

print(f'\nFinal Hierarchical Clustering Results ({best_linkage} linkage):')
print(f'  Silhouette Score: {sil_hier:.4f}')
print(f'  Davies-Bouldin Index: {db_hier:.4f}')
print(f'  Calinski-Harabasz Index: {ch_hier:.2f}')

In [ ]:
# Create Dendrogram (MANDATORY visualization per rubric)
print('Creating dendrogram...')

# For large datasets, sample for visualization
sample_size = min(100, X_scaled.shape[0])
if X_scaled.shape[0] > sample_size:
    print(f'Sampling {sample_size} points for dendrogram (dataset too large)')
    indices = np.random.choice(X_scaled.shape[0], sample_size, replace=False)
    X_sample = X_scaled[indices]
else:
    X_sample = X_scaled

# Create linkage matrix
linkage_matrix = linkage(X_sample, method=best_linkage)

# Plot dendrogram
plt.figure(figsize=(14, 7))
dendrogram(linkage_matrix, truncate_mode='level', p=5)
plt.xlabel('Sample Index', fontsize=12, fontweight='bold')
plt.ylabel('Distance', fontsize=12, fontweight='bold')
plt.title(f'Hierarchical Clustering Dendrogram ({best_linkage.capitalize()} Linkage)', 
          fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Dendrogram created')

## 5. Algorithm Comparison

Compare K-Means and Hierarchical clustering performance.

In [ ]:
# Create comparison table
comparison_data = {
    'K-Means': {
        'Silhouette Score': sil_kmeans,
        'Davies-Bouldin Index': db_kmeans,
        'Calinski-Harabasz Index': ch_kmeans
    },
    f'Hierarchical ({best_linkage})': {
        'Silhouette Score': sil_hier,
        'Davies-Bouldin Index': db_hier,
        'Calinski-Harabasz Index': ch_hier
    }
}

comparison_df = pd.DataFrame(comparison_data).T
comparison_df = comparison_df.round(4)

print('='*70)
print('CLUSTERING ALGORITHM COMPARISON')
print('='*70)
display(comparison_df)

print('\nMetric interpretation:')
print('  - Silhouette Score: HIGHER is better (range: -1 to 1)')
print('  - Davies-Bouldin Index: LOWER is better')
print('  - Calinski-Harabasz Index: HIGHER is better')

# Determine better algorithm
if sil_kmeans > sil_hier:
    print(f'\nOverall better algorithm: K-Means (higher silhouette score)')
else:
    print(f'\nOverall better algorithm: Hierarchical (higher silhouette score)')

## 6. Cluster Visualization with PCA

**Per rubric:** "Cluster visualisation via PCA (2 principal components) is MANDATORY" 

Visualize clusters in 2D space using PCA dimensionality reduction.

In [ ]:
# PCA for visualization (MANDATORY per rubric)
print('Performing PCA for 2D visualization...')
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f'Explained variance:')
print(f'  PC1: {pca.explained_variance_ratio_[0]*100:.2f}%')
print(f'  PC2: {pca.explained_variance_ratio_[1]*100:.2f}%')
print(f'  Total: {sum(pca.explained_variance_ratio_)*100:.2f}%')

In [ ]:
# Plot PCA visualizations for BOTH algorithms (MANDATORY)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means clusters
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, 
                           cmap='viridis', alpha=0.6, edgecolors='k', s=50)
# Add centroids
centroids_pca = pca.transform(kmeans_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1], 
                c='red', marker='X', s=300, edgecolors='k', linewidths=2, label='Centroids')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', 
                   fontsize=12, fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', 
                   fontsize=12, fontweight='bold')
axes[0].set_title(f'K-Means Clustering (k={optimal_k})\nSilhouette: {sil_kmeans:.4f}', 
                  fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Hierarchical clusters
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=hier_labels, 
                           cmap='plasma', alpha=0.6, edgecolors='k', s=50)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', 
                   fontsize=12, fontweight='bold')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', 
                   fontsize=12, fontweight='bold')
axes[1].set_title(f'Hierarchical Clustering ({best_linkage})\nSilhouette: {sil_hier:.4f}', 
                  fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

print('PCA visualizations created for BOTH algorithms (rubric requirement met)')

## 7. t-SNE Visualization (Encouraged)

**Per rubric:** "t-SNE visualisation is encouraged" 

t-SNE often provides better separation than PCA for complex data.

In [ ]:
# t-SNE visualization (encouraged by rubric)
print('Performing t-SNE (this may take a moment)...')

# Adjust perplexity for dataset size
perplexity = min(30, X_scaled.shape[0] - 1)

tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=perplexity)
X_tsne = tsne.fit_transform(X_scaled)

print('t-SNE complete')

In [ ]:
# Plot t-SNE visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means with t-SNE
scatter1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels, 
                           cmap='viridis', alpha=0.6, edgecolors='k', s=50)
axes[0].set_xlabel('t-SNE Component 1', fontsize=12, fontweight='bold')
axes[0].set_ylabel('t-SNE Component 2', fontsize=12, fontweight='bold')
axes[0].set_title(f'K-Means Clustering (t-SNE)', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Hierarchical with t-SNE
scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=hier_labels, 
                           cmap='plasma', alpha=0.6, edgecolors='k', s=50)
axes[1].set_xlabel('t-SNE Component 1', fontsize=12, fontweight='bold')
axes[1].set_ylabel('t-SNE Component 2', fontsize=12, fontweight='bold')
axes[1].set_title(f'Hierarchical Clustering (t-SNE)', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

print('t-SNE visualizations created')

## 8. Summary and Conclusions

### Key Findings:
- Implemented both required clustering algorithms (K-Means and Hierarchical)
- Evaluated using all 3 mandatory metrics (Silhouette, Davies-Bouldin, Calinski-Harabasz)
- Created all required visualizations (Elbow curve, Dendrogram, PCA plots)
- Added encouraged t-SNE visualization for better insights

### Review 2 Deliverables Completed:
- [x] K-Means clustering with elbow curve
- [x] Hierarchical clustering with dendrogram
- [x] All 3 mandatory metrics calculated for both algorithms
- [x] PCA 2D scatter plots for BOTH algorithms (mandatory)
- [x] t-SNE visualization (encouraged)
- [x] Linkage strategy comparison for hierarchical clustering
- [x] Clear comparison table between algorithms

### Optimal Configuration:
- **Optimal k:** {optimal_k}
- **Best linkage method:** {best_linkage}
- **Better overall algorithm:** Based on silhouette score

### Cluster Interpretation:
The clusters represent distinct groups in the data. Without ground truth labels, interpretation should be based on:
1. Feature analysis of each cluster
2. Domain knowledge
3. Business context